# 00 · Build the database

Builds the PostgreSQL database `gbc` from the saved raw/flat files and exports it to DuckDB for the dashboards.

```
data/raw + data/flat  ──►  staging  ──►  core  ──►  mart (views)  ──►  data/gbc.duckdb
                        (1:1 with files)  (dimensions + long facts)
```

The SQL lives in `src/` (so it can also run from the command line); this notebook runs it stage by stage and checks the result.
**Running all cells rebuilds the database from scratch.** Needs PostgreSQL running and the fetch scripts already run.

In [2]:
import sys, os, pathlib, time
import pandas as pd
import psycopg
from IPython.display import Code, display

ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "load"))
sys.path.insert(0, str(ROOT / "src" / "export"))
import build_db, to_duckdb

pd.set_option("display.width", 200, "display.max_columns", 30, "display.float_format", "{:,.3f}".format)
conn = psycopg.connect(build_db.DSN)

def q(sql, params=None):
    """Run a query and return a DataFrame."""
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])

def stage(label, fn):
    t = time.time(); fn(); print(f"\n✓ {label} ({time.time() - t:.1f}s)")

print("database:", q("select current_database() db, version() v").iloc[0].db, "| root:", ROOT)

database: gbc | root: /Users/hugo/Projects/name_to_decide


## 1 · Schema
Three schemas: `staging` (files loaded as-is), `core` (dimensions and long fact tables), `mart` (views). Running this **drops and recreates** all three.

In [3]:
display(Code(filename=str(ROOT / "src/load/schema.sql"), language="sql"))

-- Schema for the global business comparison database (PostgreSQL).
-- Layers: staging (1:1 with the source files) -> core (dimensions + long facts) -> mart (views).
-- Rebuilt from scratch by src/load/build_db.py.

DROP SCHEMA IF EXISTS mart CASCADE;
DROP SCHEMA IF EXISTS core CASCADE;
DROP SCHEMA IF EXISTS staging CASCADE;
CREATE SCHEMA staging;
CREATE SCHEMA core;
CREATE SCHEMA mart;

-- ───────────────────────────── STAGING ─────────────────────────────
CREATE TABLE staging.stg_sources (
    source_id TEXT, name TEXT, base_url TEXT, retrieved_at TIMESTAMP
);
CREATE TABLE staging.stg_countries_config (
    iso3 TEXT, iso2 TEXT, eurostat_code TEXT, name TEXT, region TEXT, eu_group TEXT,
    is_eu BOOLEAN, is_euro BOOLEAN, is_oecd BOOLEAN, small_firm_sample BOOLEAN
);
CREATE TABLE staging.stg_wb_countries (
    iso3 TEXT, iso2 TEXT, name TEXT, region TEXT, income_group TEXT, lat TEXT, lon TEXT
);
CREATE TABLE staging.stg_indicator_config (          -- config/indicators.yaml
    source TEXT, code TEXT, theme TEXT, snapshot_only BOOLEAN
);
-- World Bank WDI and WGI share one shape
CREATE TABLE staging.stg_worldbank (indicator_code TEXT, indicator_name TEXT, iso3 TEXT, year INT, value DOUBLE PRECISION);
CREATE TABLE staging.stg_wgi       (indicator_code TEXT, indicator_name TEXT, iso3 TEXT, year INT, value DOUBLE PRECISION);
CREATE TABLE staging.stg_imf       (indicator_code TEXT, iso3 TEXT, year INT, value DOUBLE PRECISION);
CREATE TABLE staging.stg_imf_indicators (code TEXT, label TEXT, unit TEXT, description TEXT);
CREATE TABLE staging.stg_eurostat_datasets (dataset TEXT, label TEXT, theme TEXT);
CREATE TABLE staging.stg_eurostat (
    dataset TEXT, iso3 TEXT, geo TEXT, period TEXT, value DOUBLE PRECISION, flag TEXT, dims JSONB
);
CREATE TABLE staging.stg_yf_seed (
    ticker TEXT, iso3 TEXT, name TEXT, sector TEXT, industry TEXT, currency TEXT, market_cap DOUBLE PRECISION
);
CREATE TABLE staging.stg_yf_meta (
    ticker TEXT, iso3 TEXT, ok BOOLEAN, longname TEXT, country TEXT, sector TEXT, industry TEXT,
    currency TEXT, financialcurrency TEXT, marketcap DOUBLE PRECISION, exchange TEXT,
    fulltimeemployees DOUBLE PRECISION
);
CREATE TABLE staging.stg_yf_financials (
    ticker TEXT, iso3 TEXT, statement TEXT, period_end DATE, item TEXT, value DOUBLE PRECISION
);
CREATE TABLE staging.stg_esef_index (
    fxo_id TEXT, country TEXT, lei TEXT, entity_name TEXT, json_url TEXT, date_added TIMESTAMP
);
CREATE TABLE staging.stg_esef_facts (
    fxo_id TEXT, lei TEXT, entity_name TEXT, country TEXT, concept TEXT,
    period_start DATE, period_end DATE, unit TEXT, value DOUBLE PRECISION, iso3 TEXT
);

-- ───────────────────────────── CORE ─────────────────────────────
CREATE TABLE core.dim_source (
    source_id    TEXT PRIMARY KEY,           -- WB, WGI, IMF, ESTAT, YF, ESEF
    name         TEXT,
    base_url     TEXT,
    retrieved_at TIMESTAMP
);

CREATE TABLE core.dim_country (
    iso3              TEXT PRIMARY KEY,
    iso2              TEXT,
    eurostat_code     TEXT,                  -- NULL outside the EU; Greece is EL
    name              TEXT,
    region            TEXT,
    eu_group          TEXT,                  -- Nordic / Western / Southern / Eastern (EU only)
    income_group      TEXT,
    is_eu             BOOLEAN,
    is_euro           BOOLEAN,
    is_oecd           BOOLEAN,
    small_firm_sample BOOLEAN,               -- market too small to reach ~50 firms
    lat               DOUBLE PRECISION,
    lon               DOUBLE PRECISION
);

CREATE TABLE core.dim_indicator (
    indicator_id  TEXT PRIMARY KEY,          -- 'WB:SL.UEM.TOTL.ZS', 'ESTAT:une_rt_a:age=Y15-74|sex=T|unit=PC_ACT'
    source_id     TEXT NOT NULL REFERENCES core.dim_source,
    source_code   TEXT NOT NULL,             -- indicator code, or dataset code for Eurostat
    name          TEXT,
    description   TEXT,
    theme         TEXT,
    unit          TEXT,
    snapshot_only BOOLEAN DEFAULT FALSE,
    filters       JSONB                 

In [4]:
stage("schema created", lambda: build_db.run_sql(conn, ROOT / "src/load/schema.sql"))

running src/load/schema.sql

✓ schema created (0.0s)


## 2 · Staging
Each source file is bulk-loaded into its own staging table with no transformation (`COPY`):
World Bank WDI / WGI, IMF, Eurostat, yfinance (seed, metadata, statements), ESEF (index, facts), plus country metadata and the indicator config.

In [5]:
stage("staging loaded", lambda: build_db.load_staging(conn))

loading staging
  staging.stg_sources: 6 rows
  staging.stg_countries_config: 34 rows
  staging.stg_wb_countries: 34 rows
  staging.stg_indicator_config: 43 rows
  staging.stg_worldbank: 14,105 rows
  staging.stg_wgi: 2,730 rows
  staging.stg_imf: 2,719 rows
  staging.stg_imf_indicators: 132 rows
  staging.stg_eurostat: 7,277 rows
  staging.stg_eurostat_datasets: 12 rows
  staging.stg_yf_seed: 2,665 rows
  staging.stg_yf_meta: 2,665 rows
  staging.stg_yf_financials: 1,190,780 rows
  staging.stg_esef_index: 11,333 rows
  staging.stg_esef_facts: 226,668 rows

✓ staging loaded (4.3s)


In [6]:
q("""
SELECT table_name, (xpath('/row/c/text()', query_to_xml('select count(*) as c from staging.'||table_name, false, true, '')))[1]::text::bigint AS rows
FROM information_schema.tables WHERE table_schema='staging' ORDER BY 1
""")

,table_name,rows
0,stg_countries_config,34
1,stg_esef_facts,226668
2,stg_esef_index,11333
3,stg_eurostat,7277
4,stg_eurostat_datasets,12
5,stg_imf,2719
6,stg_imf_indicators,132
7,stg_indicator_config,43
8,stg_sources,6
9,stg_wb_countries,34


## 3 · Core
Staging → core. Rules applied here:

- Only countries in `dim_country` (from `config/countries.csv`) are loaded.
- **Indicators**: one row per series. Eurostat series are identified by dataset + dimension values (e.g. `ESTAT:une_rt_a:age=Y15-74|sex=T|unit=PC_ACT`), kept in `filters`.
- **Companies**: `company_id` is `YF:<ticker>` or `ESEF:<lei>`; both sources share `fact_company_financial`.
- **Fiscal year start**: reported by ESEF where available; otherwise derived as *end − 1 year + 1 day* and flagged `fy_start_derived`.
- **ESEF**: annual durations and all instants only; one reporting currency per company; if a fact appears in several filings (prior-year comparatives) the latest filing wins.

In [7]:
display(Code(filename=str(ROOT / "src/transform/core.sql"), language="sql"))

-- staging -> core. Only countries in dim_country are loaded (config/countries.csv decides the scope).

INSERT INTO core.dim_source SELECT source_id, name, base_url, retrieved_at FROM staging.stg_sources;

INSERT INTO core.dim_country
SELECT c.iso3, c.iso2, NULLIF(c.eurostat_code, ''), c.name, c.region, NULLIF(c.eu_group, ''),
       w.income_group, c.is_eu, c.is_euro, c.is_oecd, c.small_firm_sample,
       NULLIF(w.lat, '')::double precision, NULLIF(w.lon, '')::double precision
FROM staging.stg_countries_config c
LEFT JOIN staging.stg_wb_countries w USING (iso3);

-- ── indicators ──────────────────────────────────────────────────────────────
INSERT INTO core.dim_indicator (indicator_id, source_id, source_code, name, theme, snapshot_only)
SELECT 'WB:' || s.indicator_code, 'WB', s.indicator_code, MAX(s.indicator_name), MAX(k.theme),
       COALESCE(BOOL_OR(k.snapshot_only), FALSE)
FROM staging.stg_worldbank s
LEFT JOIN staging.stg_indicator_config k ON k.source = 'WB' AND k.code = s.indicator_code
GROUP BY s.indicator_code;

INSERT INTO core.dim_indicator (indicator_id, source_id, source_code, name, theme, snapshot_only)
SELECT 'WGI:' || s.indicator_code, 'WGI', s.indicator_code, MAX(s.indicator_name), MAX(k.theme),
       COALESCE(BOOL_OR(k.snapshot_only), FALSE)
FROM staging.stg_wgi s
LEFT JOIN staging.stg_indicator_config k ON k.source = 'WGI' AND k.code = s.indicator_code
GROUP BY s.indicator_code;

INSERT INTO core.dim_indicator (indicator_id, source_id, source_code, name, description, theme, unit)
SELECT 'IMF:' || d.indicator_code, 'IMF', d.indicator_code, i.label, i.description,
       (SELECT k.theme FROM staging.stg_indicator_config k WHERE k.source = 'IMF' AND k.code = d.indicator_code),
       i.unit
FROM (SELECT DISTINCT indicator_code FROM staging.stg_imf) d
LEFT JOIN staging.stg_imf_indicators i ON i.code = d.indicator_code;

INSERT INTO core.dim_indicator (indicator_id, source_id, source_code, name, description, theme, unit, filters)
SELECT 'ESTAT:' || e.dataset || ':' || e.dims_key, 'ESTAT', e.dataset, d.label, e.dims_key, d.theme,
       e.dims ->> 'unit', e.dims
FROM (SELECT DISTINCT dataset, dims,
             (SELECT string_agg(k || '=' || v, '|' ORDER BY k) FROM jsonb_each_text(dims) AS t(k, v)) AS dims_key
      FROM staging.stg_eurostat) e
JOIN staging.stg_eurostat_datasets d USING (dataset);

-- ── country-year facts ──────────────────────────────────────────────────────
INSERT INTO core.fact_country_year (iso3, indicator_id, period, year, value)
SELECT iso3, 'WB:' || indicator_code, year::text, year, value
FROM staging.stg_worldbank WHERE value IS NOT NULL AND iso3 IN (SELECT iso3 FROM core.dim_country);

INSERT INTO core.fact_country_year (iso3, indicator_id, period, year, value)
SELECT iso3, 'WGI:' || indicator_code, year::text, year, value
FROM staging.stg_wgi WHERE value IS NOT NULL AND iso3 IN (SELECT iso3 FROM core.dim_country);

INSERT INTO core.fact_country_year (iso3, indicator_id, period, year, value)
SELECT iso3, 'IMF:' || indicator_code, year::text, year, value
FROM staging.stg_imf WHERE value IS NOT NULL AND iso3 IN (SELECT iso3 FROM core.dim_country);

INSERT INTO core.fact_country_year (iso3, indicator_id, period, year, value, flag)
SELECT iso3,
       'ESTAT:' || dataset || ':' || (SELECT string_agg(k || '=' || v, '|' ORDER BY k) FROM jsonb_each_text(dims) AS t(k, v)),
       period, LEFT(period, 4)::int, value, NULLIF(flag, '')
FROM staging.stg_eurostat
WHERE value IS NOT NULL AND iso3 IN (SELECT iso3 FROM core.dim_country);

-- ── companies ───────────────────────────────────────────────────────────────
INSERT INTO core.dim_company
SELECT DISTINCT ON (s.ticker)
       'YF:' || s.ticker, 'YF', s.ticker, COALESCE(m.longname, s.name), s.iso3,
       COALESCE(m.sector, s.sector), COALESCE(m.industry, s.industry), m.exchange,
       COALESCE(m.currency, s.currency), m.financialcurrency,
       COALESCE(m.marketcap, s.market_cap), m.fulltimeemployees
FROM staging.stg_yf_seed s
LEFT JOIN sta

In [8]:
stage("core built", lambda: build_db.run_sql(conn, ROOT / "src/transform/core.sql"))

running src/transform/core.sql

✓ core built (17.2s)


## 4 · Marts (views)

In [9]:
display(Code(filename=str(ROOT / "src/transform/marts.sql"), language="sql"))

-- Views for dashboards and analysis. No data is stored here.

-- latest non-null value per country and indicator (rankings, maps)
CREATE VIEW mart.v_country_latest AS
SELECT DISTINCT ON (f.iso3, f.indicator_id)
       f.iso3, f.indicator_id, i.name AS indicator_name, i.theme, f.period, f.year, f.value
FROM core.fact_country_year f
JOIN core.dim_indicator i USING (indicator_id)
ORDER BY f.iso3, f.indicator_id, f.period DESC;

-- share of years 2012-2024 with data, per country and indicator (coverage heatmap).
-- Eurostat series only apply to EU countries, so non-EU countries are left out of that grid.
CREATE VIEW mart.v_coverage AS
WITH grid AS (
    SELECT c.iso3, i.indicator_id, i.source_id, i.name AS indicator_name, i.theme
    FROM core.dim_country c
    CROSS JOIN core.dim_indicator i
    WHERE i.source_id <> 'ESTAT' OR c.is_eu
), have AS (
    SELECT iso3, indicator_id, COUNT(DISTINCT year) AS n_years, MIN(year) AS first_year, MAX(year) AS last_year
    FROM core.fact_country_year
    WHERE year BETWEEN 2012 AND 2024
    GROUP BY iso3, indicator_id
)
SELECT g.*, COALESCE(h.n_years, 0) AS n_years, h.first_year, h.last_year,
       ROUND(100.0 * COALESCE(h.n_years, 0) / 13, 1) AS pct_years
FROM grid g LEFT JOIN have h USING (iso3, indicator_id);

-- one row per company and fiscal year, standard metrics side by side.
-- fiscal_year = calendar year in which most of the fiscal year falls (midpoint rule).
-- Ratios are raw (not winsorised); revenue_growth needs the previous year to be ~12 months earlier.
CREATE VIEW mart.v_company_year AS
WITH m AS (
    SELECT f.company_id, f.fiscal_year_start, f.period_end, mm.metric, f.value, f.currency, mm.priority
    FROM core.fact_company_financial f
    JOIN core.dim_company c USING (company_id)
    JOIN core.company_metric_map mm
      ON mm.source_id = c.source_id AND mm.statement = f.statement AND mm.item = f.item
), best AS (
    SELECT DISTINCT ON (company_id, period_end, metric) *
    FROM m ORDER BY company_id, period_end, metric, priority, fiscal_year_start
), wide AS (
    SELECT company_id, period_end, MIN(fiscal_year_start) AS fiscal_year_start, MAX(currency) AS currency,
           MAX(value) FILTER (WHERE metric = 'revenue')          AS revenue,
           MAX(value) FILTER (WHERE metric = 'operating_income') AS operating_income,
           MAX(value) FILTER (WHERE metric = 'net_income')       AS net_income,
           MAX(value) FILTER (WHERE metric = 'total_assets')     AS total_assets,
           MAX(value) FILTER (WHERE metric = 'equity')           AS equity
    FROM best GROUP BY company_id, period_end
), lagged AS (
    SELECT w.*,
           LAG(revenue)    OVER (PARTITION BY company_id ORDER BY period_end) AS prev_revenue,
           LAG(period_end) OVER (PARTITION BY company_id ORDER BY period_end) AS prev_period_end
    FROM wide w
)
SELECT l.company_id, c.source_id, c.name, c.iso3, c.sector,
       EXTRACT(YEAR FROM l.fiscal_year_start + (l.period_end - l.fiscal_year_start) / 2)::int AS fiscal_year,
       l.fiscal_year_start, l.period_end, l.currency,
       l.revenue, l.operating_income, l.net_income, l.total_assets, l.equity,
       l.net_income / NULLIF(l.total_assets, 0)      AS roa,
       l.operating_income / NULLIF(l.revenue, 0)     AS operating_margin,
       CASE WHEN l.period_end - l.prev_period_end BETWEEN 350 AND 380
            THEN l.revenue / NULLIF(l.prev_revenue, 0) - 1 END AS revenue_growth
FROM lagged l JOIN core.dim_company c USING (company_id);

-- how many companies each country has, per source, with usable data (assets and net income)
CREATE VIEW mart.v_company_coverage AS
SELECT c.iso3, y.source_id,
       COUNT(DISTINCT y.company_id) AS companies,
       COUNT(*)                     AS firm_years,
       ROUND(COUNT(*)::numeric / COUNT(DISTINCT y.company_id), 1) AS years_per_company
FROM mart.v_company_year y
JOIN core.dim_company c USING (company_id)
WHERE y.net_income IS NOT NULL AND y.total_assets IS NOT NULL
GROUP BY c.iso3, y.sourc

In [10]:
stage("marts built", lambda: build_db.run_sql(conn, ROOT / "src/transform/marts.sql"))
conn.commit()

running src/transform/marts.sql

✓ marts built (0.0s)


## 5 · Checks

In [11]:
q("""
SELECT 'dim_country' t, COUNT(*) n FROM core.dim_country UNION ALL
SELECT 'dim_indicator', COUNT(*) FROM core.dim_indicator UNION ALL
SELECT 'fact_country_year', COUNT(*) FROM core.fact_country_year UNION ALL
SELECT 'dim_company', COUNT(*) FROM core.dim_company UNION ALL
SELECT 'fact_company_financial', COUNT(*) FROM core.fact_company_financial
""")

,t,n
0,dim_country,34
1,dim_indicator,77
2,fact_country_year,24918
3,dim_company,3902
4,fact_company_financial,1282447


In [12]:
# indicators and facts by source
q("""
SELECT i.source_id, COUNT(DISTINCT i.indicator_id) indicators, COUNT(f.*) facts, COUNT(DISTINCT f.iso3) countries
FROM core.dim_indicator i LEFT JOIN core.fact_country_year f USING (indicator_id)
GROUP BY 1 ORDER BY 1
""")

,source_id,indicators,facts,countries
0,ESTAT,34,6930,21
1,IMF,6,2641,34
2,WB,31,12695,34
3,WGI,6,2652,34


In [13]:
# integrity: each of these should return 0
checks = {
  "indicators without facts": "SELECT COUNT(*) FROM core.dim_indicator i WHERE NOT EXISTS (SELECT 1 FROM core.fact_country_year f WHERE f.indicator_id=i.indicator_id)",
  "companies without financials": "SELECT COUNT(*) FROM core.dim_company c WHERE NOT EXISTS (SELECT 1 FROM core.fact_company_financial f WHERE f.company_id=c.company_id)",
  "fiscal_year_start after period_end": "SELECT COUNT(*) FROM core.fact_company_financial WHERE fiscal_year_start > period_end",
  "income rows with fiscal year outside 360-370 days": "SELECT COUNT(*) FROM core.fact_company_financial WHERE statement='income' AND (period_end - fiscal_year_start) NOT BETWEEN 360 AND 370",
  "facts for countries not in dim_country": "SELECT COUNT(*) FROM staging.stg_worldbank s WHERE value IS NOT NULL AND iso3 NOT IN (SELECT iso3 FROM core.dim_country) AND iso3 IN (SELECT iso3 FROM staging.stg_countries_config)",
}
pd.Series({k: q(v).iloc[0, 0] for k, v in checks.items()}, name="violations")

indicators without facts                             0
companies without financials                         0
fiscal_year_start after period_end                   0
income rows with fiscal year outside 360-370 days    0
facts for countries not in dim_country               0
Name: violations, dtype: int64

In [14]:
# fiscal year start: reported vs derived
q("""
SELECT c.source_id, f.fy_start_derived AS derived, COUNT(*) rows
FROM core.fact_company_financial f JOIN core.dim_company c USING (company_id)
GROUP BY 1, 2 ORDER BY 1, 2
""")

,source_id,derived,rows
0,ESEF,False,85296
1,ESEF,True,6371
2,YF,True,1190780


In [15]:
# companies per country and source (usable = has both net income and total assets)
cov = q("SELECT iso3, source_id, companies FROM mart.v_company_coverage")
tbl = cov.pivot(index="iso3", columns="source_id", values="companies").fillna(0).astype(int)
tbl["total_if_no_overlap"] = tbl.sum(axis=1)
tbl.sort_values("total_if_no_overlap")

source_id,ESEF,YF,total_if_no_overlap
iso3,,,
LVA,13,10,23
EST,16,18,34
CZE,22,17,39
IRL,0,43,43
LTU,27,22,49
HUN,39,21,60
PRT,39,32,71
DEU,0,79,79
CHL,0,92,92


In [16]:
# spot check: Toyota (March year end) and one ESEF company
q("""
SELECT name, fiscal_year, fiscal_year_start, period_end, currency, revenue, net_income, total_assets, roa, revenue_growth
FROM mart.v_company_year
WHERE company_id = 'YF:7203.T' OR (source_id='ESEF' AND name LIKE 'AB Traction%')
ORDER BY name, period_end
""")

,name,fiscal_year,fiscal_year_start,period_end,currency,revenue,net_income,total_assets,roa,revenue_growth
0,AB Traction,2020,2020-01-01,2020-12-31,SEK,"-7,500,000.000","-18,500,000.000","2,921,800,000.000",-0.006,NaN
1,AB Traction,2021,2021-01-01,2021-12-31,SEK,"1,353,700,000.000","1,343,700,000.000","4,237,500,000.000",0.317,-181.493
2,AB Traction,2022,2022-01-01,2022-12-31,SEK,"-362,100,000.000","-377,800,000.000","3,792,500,000.000",-0.100,-1.267
3,AB Traction,2023,2023-01-01,2023-12-31,SEK,"284,400,000.000","269,800,000.000","3,933,600,000.000",0.069,-1.785
4,AB Traction,2024,2024-01-01,2024-12-31,SEK,"490,400,000.000","477,100,000.000","4,252,100,000.000",0.112,0.724
5,Toyota Motor Corporation,2022,2022-04-01,2023-03-31,JPY,"37,154,298,000,000.000","2,451,318,000,000.000","74,303,180,000,000.000",0.033,NaN
6,Toyota Motor Corporation,2023,2023-04-01,2024-03-31,JPY,"45,095,325,000,000.000","4,944,933,000,000.000","90,114,296,000,000.000",0.055,0.214
7,Toyota Motor Corporation,2024,2024-04-01,2025-03-31,JPY,"48,036,704,000,000.000","4,765,086,000,000.000","93,601,350,000,000.000",0.051,0.065
8,Toyota Motor Corporation,2025,2025-04-01,2026-03-31,JPY,"50,684,952,000,000.000","3,848,098,000,000.000","105,522,331,000,000.000",0.036,0.055


In [17]:
# firm-years by fiscal year and source
q("""
SELECT fiscal_year, source_id, COUNT(*) firm_years
FROM mart.v_company_year WHERE net_income IS NOT NULL AND total_assets IS NOT NULL
GROUP BY 1, 2 ORDER BY 1, 2
""").pivot(index="fiscal_year", columns="source_id", values="firm_years")

source_id,ESEF,YF
fiscal_year,,
2019,70.000,NaN
2020,626.000,NaN
2021,930.000,89.000
2022,"1,077.000","2,530.000"
2023,"1,077.000","2,548.000"
2024,970.000,"2,547.000"
2025,562.000,"2,507.000"
2026,NaN,2.000


## 6 · Export to DuckDB
Copies the core tables and materialises the mart views into `data/gbc.duckdb`, which the dashboards read. The file is rebuilt from scratch and never edited by hand.

In [18]:
stage("exported to DuckDB", to_duckdb.main)

  dim_source: 6 rows
  dim_country: 34 rows
  dim_indicator: 77 rows
  fact_country_year: 24,918 rows
  dim_company: 3,902 rows
  fact_company_financial: 1,282,447 rows
  company_metric_map: 14 rows
  v_country_latest: 2,127 rows
  v_coverage: 2,176 rows
  v_company_year: 17,608 rows
  v_company_coverage: 53 rows

Wrote /Users/hugo/Projects/name_to_decide/data/gbc.duckdb (22.8 MB)

✓ exported to DuckDB (2.4s)


In [19]:
import duckdb
d = duckdb.connect(str(to_duckdb.OUT), read_only=True)
d.sql("SELECT table_name, estimated_size AS rows FROM duckdb_tables() ORDER BY 1").df()

,table_name,rows
0,company_metric_map,14
1,dim_company,3902
2,dim_country,34
3,dim_indicator,77
4,dim_source,6
5,fact_company_financial,1282447
6,fact_country_year,24918
7,v_company_coverage,53
8,v_company_year,17608
9,v_country_latest,2127


In [20]:
conn.close(); d.close()
print("done")

done
